In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
from IPython.display import display

TRAIN_PATH = "Devex_train.csv"
TEST_PATH = "Devex_test_questions.csv"

train_df = pd.read_csv(TRAIN_PATH, encoding="latin-1")
test_df = pd.read_csv(TEST_PATH, encoding="latin-1")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# snip display of the train and test datasets
print("\nTrain data (first 2 rows):")
display(train_df.head(2))

print("\nTest data (first 2 rows):")
display(test_df.head(2))

In [ ]:
# Parsing Labels
LABEL_COLS = [f"Label {i}" for i in range(1, 13)]


def extract_labels(row):
    """Return active SDG indicator strings for one row (ignore NA/NaN/empty)."""
    labels = []
    for col in LABEL_COLS:
        value = row[col]
        if pd.isna(value):
            continue
        value = str(value).strip()
        if value and value.upper() != "NA":
            labels.append(value)
    return labels


train_df["labels"] = train_df.apply(extract_labels, axis=1)
train_df["n_labels"] = train_df["labels"].apply(len)

# Build fixed label vocabulary from training set only
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(train_df["labels"])
label_names = mlb.classes_

print(f"Number of unique SDG 3 indicators: {len(label_names)}")
print(f"Label matrix shape (samples × labels): {Y.shape}")
print(f"Labels per document — min: {train_df['n_labels'].min()}, "
      f"max: {train_df['n_labels'].max()}, "
      f"mean: {train_df['n_labels'].mean():.2f}")

# Shared split settings for preprocessing ablation, baseline, and experiments 1–10
RANDOM_STATE = 42
TEST_SIZE = 0.2
y = Y

display(train_df[["Unique ID", "Type", "n_labels", "labels"]].head(3))
